# 🎭 Nuance Talking Avatar — Real-Time Cloud GPU Runner (LivePortrait)

This notebook runs the **Nuance-style Talking Avatar** system with **NVIDIA T4 GPU acceleration** for free on Google Colab.

### Capabilities:
- **High-Speed Real-Time Synthesis**: ~30–35 FPS inference (~25ms/frame) powered by NVIDIA T4 CUDA acceleration.
- **Active Listening & Micro-Expressions**: Real-time controls for head pitch/yaw/roll micro-nods, eye retargeting, eyeball saccades, smile, and lip articulation.
- **Turnkey Public URL**: Launches a live `gradio.live` link for interactive browser control.

> **Step 1:** Make sure your Colab runtime is set to **T4 GPU**:
> Click **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Verify GPU Hardware Acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    raise SystemError("GPU not detected! Please go to Runtime > Change runtime type > select T4 GPU.")

In [ ]:
# 2. Clone LivePortrait Repository & Setup Working Directory
import os
%cd /content
if not os.path.exists("/content/LivePortrait"):
    !git clone https://github.com/samuelQUANSAH/nuance-talking-avatar.git /content/LivePortrait
%cd /content/LivePortrait

In [ ]:
# 3. Install GPU Dependencies (with pinned compatibility)
!pip install -q onnx onnxruntime-gpu opencv-python-headless tyro "gradio>=4.38.0,<5.0.0" "huggingface_hub<0.25.0" pyyaml imageio imageio-ffmpeg scipy pykalman

In [ ]:
# 4. Download Pretrained Neural Weights (~2.5GB in ~15-20s on Colab)
import os
from huggingface_hub import snapshot_download

os.makedirs("pretrained_weights", exist_ok=True)
print("Streaming foundation weights from KlingTeam/LivePortrait...")
snapshot_download(
    repo_id="KlingTeam/LivePortrait",
    local_dir="pretrained_weights",
    ignore_patterns=["*.git*", "README.md", "docs/*", "*animals*"],
    local_dir_use_symlinks=False
)
print("Weights successfully initialized!")

In [ ]:
# 5. Setup Avatar Portrait & Apply Compatibility Patches
import os

# Ensure HfFolder patch in app.py
if os.path.exists("app.py"):
    with open("app.py", "r") as f:
        app_code = f.read()
    patch = '''import huggingface_hub
if not hasattr(huggingface_hub, "HfFolder"):
    class HfFolder:
        @classmethod
        def get_token(cls):
            return huggingface_hub.get_token()
    huggingface_hub.HfFolder = HfFolder
'''
    if "huggingface_hub.HfFolder = HfFolder" not in app_code:
        app_code = patch + app_code
        with open("app.py", "w") as f:
            f.write(app_code)
        print("[✓] Injected HfFolder compatibility patch into app.py")

# Inject user avatar as primary preset in app.py if not present
if os.path.exists("app.py"):
    with open("app.py", "r") as f:
        code = f.read()
    if 'avatar_source.jpg' not in code and 'data_examples_i2v = [' in code:
        code = code.replace(
            'data_examples_i2v = [',
            'data_examples_i2v = [\n    [osp.join(example_portrait_dir, "avatar_source.jpg"), osp.join(example_video_dir, "d0.mp4"), True, True, True, False],'
        )
        with open("app.py", "w") as f:
            f.write(code)
        print("[✓] Injected avatar portrait as primary preset in app.py")

# Patch retargeting key error handling in live_portrait_pipeline.py
pipeline_file = "src/live_portrait_pipeline.py"
if os.path.exists(pipeline_file):
    with open(pipeline_file, "r") as f:
        p_code = f.read()
    if "c_d_eyes_lst" in p_code and "driving_template_dct.get('c_eyes_lst'" not in p_code:
        p_code = p_code.replace(
            "c_d_eyes_lst = driving_template_dct['c_d_eyes_lst']",
            "c_d_eyes_lst = driving_template_dct.get('c_eyes_lst', driving_template_dct.get('c_d_eyes_lst', None))"
        ).replace(
            "c_d_lip_lst = driving_template_dct['c_d_lip_lst']",
            "c_d_lip_lst = driving_template_dct.get('c_lip_lst', driving_template_dct.get('c_d_lip_lst', None))"
        )
        with open(pipeline_file, "w") as f:
            f.write(p_code)
        print("[✓] Applied template retargeting key compatibility patch.")
print("Setup complete!")

In [ ]:
# 6. Launch the Interactive Avatar Web Application (with Public Gradio Share Tunnel)
# Open the generated https://xxxx.gradio.live link in your browser to animate your avatar!
!python app.py --share